In [1]:
%%bash

wget -O "./protein/Apis_mellifera.faa.gz" "https://ftp.ensemblgenomes.ebi.ac.uk/pub/metazoa/current/fasta/apis_mellifera/pep/Apis_mellifera.Amel_HAv3.1.pep.all.fa.gz"
gunzip "./protein/Apis_mellifera.faa.gz"

--2026-11-01 13:27:22--  https://ftp.ensemblgenomes.ebi.ac.uk/pub/metazoa/current/fasta/apis_mellifera/pep/Apis_mellifera.Amel_HAv3.1.pep.all.fa.gz
Resolving ftp.ensemblgenomes.ebi.ac.uk (ftp.ensemblgenomes.ebi.ac.uk)... 193.62.193.161
Connecting to ftp.ensemblgenomes.ebi.ac.uk (ftp.ensemblgenomes.ebi.ac.uk)|193.62.193.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5385788 (5.1M) [application/x-gzip]
Saving to: ‘./protein/Apis_mellifera.faa.gz’

     0K .......... .......... .......... .......... ..........  0%  240K 22s
    50K .......... .......... .......... .......... ..........  1%  152K 28s
   100K .......... .......... .......... .......... ..........  2% 18.2M 18s
   150K .......... .......... .......... .......... ..........  3% 1.76M 14s
   200K .......... .......... .......... .......... ..........  4%  326K 14s
   250K .......... .......... .......... .......... ..........  5%  106M 12s
   300K .......... .......... .......... .......... .....

确定 adata 的 gene_id 与蛋白组中的 header 都是 GTF 中特定列的子集：

In [2]:
import scanpy as sc
import pandas as pd
from pyfaidx import Fasta

/home/liuzy/Software/miniconda3/envs/saturn/lib/python3.9/site-packages/torch/torch_version.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


In [3]:
Amel_gtf = pd.read_csv("./metadata/Apis_mellifera.tsv", sep="\t")
#! Apis_mellifera.tsv: see https://github.com/zliu90693/anno_cells/tree/main/.subprojects/MetaNeighbor/metadata/Apis_mellifera.tsv

In [4]:
Amel_gtf_gid = set(Amel_gtf["gene_id"])
Amel_gtf_pid = set(Amel_gtf["protein_id"])

In [ ]:
Amel_adata = sc.read_h5ad("./matrix/Amel_wll.h5ad")
#! Amel_wll.h5ad，see https://github.com/zliu90693/anno_cells/blob/main/.subprojects/MetaNeighbor/01_01-get_n_check_data-Amel.ipynb

In [9]:
Amel_adata

AnnData object with n_obs × n_vars = 115169 × 12292
    obs: 'sample', 'amel_key', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'f', 'n', 'q', 'integrated_snn_res.1', 'seurat_clusters', 'type', 'cell_type'

In [10]:
Amel_adata_gid = set(Amel_adata.var.index)

In [ ]:
Amel_protein = Fasta("./protein/Apis_mellifera.faa")
Amel_protein_pid = set(header for header in Amel_protein.keys())

In [12]:
print(len(Amel_gtf_gid), len(Amel_adata_gid), len(Amel_gtf_gid & Amel_adata_gid))

12398 12292 12292


In [19]:
print(len(Amel_gtf_pid), len(Amel_protein_pid), len(Amel_gtf_pid & Amel_protein_pid))

23471 23470 23470


构建 protein-gene 对应关系

In [21]:
import json
import pandas as pd

Amel_gtf = pd.read_csv("metadata/Apis_mellifera.tsv", sep="\t", low_memory=False)

m = Amel_gtf.dropna(subset=["gene_id", "protein_id"])[["gene_id", "protein_id"]]
m = m.drop_duplicates()
m["gene_id"] = m["gene_id"].astype(str)
m["protein_id"] = m["protein_id"].astype(str)

gene_to_proteins = {}
for gene, sub in m.groupby("gene_id"):
    gene_to_proteins[gene] = sorted(set(sub["protein_id"]))

with open("./gene_protein_map/Amel.gene_id_to_protein_ID.json", "w") as f:
    json.dump(gene_to_proteins, f, indent=4, sort_keys=True)

生成 SATURN-ready 的 h5ad

In [1]:
import scanpy as sc
import numpy as np

/home/liuzy/Software/miniconda3/envs/saturn/lib/python3.9/site-packages/torch/torch_version.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


In [ ]:
Amel = sc.read_h5ad("matrix/Amel_wll.h5ad") # Amel 的 X 是原始技术矩阵，不用替换为其他 layer
Amel.X = Amel.X.astype(np.float32) # 

In [3]:
Amel.obs

,sample,amel_key,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,f,n,q,integrated_snn_res.1,seurat_clusters,type,cell_type
AAACCCAAGAGCCTGA-1-GSM5590453_scFoager1,GSM5590453_scFoager1,F1_sc_AAACCCAAGAGCCTGA-1,F1,4938.0,2284.0,5.548805,F,NaN,NaN,24.0,24.0,F,UNs
AAACCCAAGAGTCACG-1-GSM5590453_scFoager1,GSM5590453_scFoager1,F1_sc_AAACCCAAGAGTCACG-1,F1,2181.0,1277.0,7.473636,F,NaN,NaN,9.0,9.0,F,KCs
AAACCCAAGCACTCGC-1-GSM5590453_scFoager1,GSM5590453_scFoager1,F1_sc_AAACCCAAGCACTCGC-1,F1,5417.0,2227.0,12.017722,F,NaN,NaN,6.0,6.0,F,OPNs
AAACCCAAGCGCACAA-1-GSM5590453_scFoager1,GSM5590453_scFoager1,F1_sc_AAACCCAAGCGCACAA-1,F1,5231.0,2255.0,6.652648,F,NaN,NaN,25.0,25.0,F,Glial cells
AAACCCAAGGCCCGTT-1-GSM5590453_scFoager1,GSM5590453_scFoager1,F1_sc_AAACCCAAGGCCCGTT-1,F1,6388.0,2624.0,5.541641,F,NaN,NaN,34.0,34.0,F,KCs
...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGTCCGTGGCA-1-GSM5590458_scQueen2,GSM5590458_scQueen2,Q2_sc_TTTGTTGTCCGTGGCA-1,Q2,6349.0,2577.0,5.937943,NaN,NaN,Q,3.0,3.0,Q,UNs
TTTGTTGTCGGCTTGG-1-GSM5590458_scQueen2,GSM5590458_scQueen2,Q2_sc_TTTGTTGTCGGCTTGG-1,Q2,3377.0,1682.0,6.692330,NaN,NaN,Q,5.0,5.0,Q,KCs
TTTGTTGTCTACCCAC-1-GSM5590458_scQueen2,GSM5590458_scQueen2,Q2_sc_TTTGTTGTCTACCCAC-1,Q2,1849.0,999.0,13.142239,NaN,NaN,Q,32.0,32.0,Q,Glial cells
TTTGTTGTCTCACTCG-1-GSM5590458_scQueen2,GSM5590458_scQueen2,Q2_sc_TTTGTTGTCTCACTCG-1,Q2,5251.0,2194.0,4.532470,NaN,NaN,Q,11.0,11.0,Q,KCs


In [4]:
Amel.obs["saturn_label"] = (
    Amel.obs["cell_type"].astype(str)
    .str.replace(r"[.\|\s]+", "-", regex=True)   # Glial cells -> Glial_cells
    + "-"
    + Amel.obs["seurat_clusters"].astype(int).astype(str)
)
Amel.obs

,sample,amel_key,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,f,n,q,integrated_snn_res.1,seurat_clusters,type,cell_type,saturn_label
AAACCCAAGAGCCTGA-1-GSM5590453_scFoager1,GSM5590453_scFoager1,F1_sc_AAACCCAAGAGCCTGA-1,F1,4938.0,2284.0,5.548805,F,NaN,NaN,24.0,24.0,F,UNs,UNs-24
AAACCCAAGAGTCACG-1-GSM5590453_scFoager1,GSM5590453_scFoager1,F1_sc_AAACCCAAGAGTCACG-1,F1,2181.0,1277.0,7.473636,F,NaN,NaN,9.0,9.0,F,KCs,KCs-9
AAACCCAAGCACTCGC-1-GSM5590453_scFoager1,GSM5590453_scFoager1,F1_sc_AAACCCAAGCACTCGC-1,F1,5417.0,2227.0,12.017722,F,NaN,NaN,6.0,6.0,F,OPNs,OPNs-6
AAACCCAAGCGCACAA-1-GSM5590453_scFoager1,GSM5590453_scFoager1,F1_sc_AAACCCAAGCGCACAA-1,F1,5231.0,2255.0,6.652648,F,NaN,NaN,25.0,25.0,F,Glial cells,Glial-cells-25
AAACCCAAGGCCCGTT-1-GSM5590453_scFoager1,GSM5590453_scFoager1,F1_sc_AAACCCAAGGCCCGTT-1,F1,6388.0,2624.0,5.541641,F,NaN,NaN,34.0,34.0,F,KCs,KCs-34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGTCCGTGGCA-1-GSM5590458_scQueen2,GSM5590458_scQueen2,Q2_sc_TTTGTTGTCCGTGGCA-1,Q2,6349.0,2577.0,5.937943,NaN,NaN,Q,3.0,3.0,Q,UNs,UNs-3
TTTGTTGTCGGCTTGG-1-GSM5590458_scQueen2,GSM5590458_scQueen2,Q2_sc_TTTGTTGTCGGCTTGG-1,Q2,3377.0,1682.0,6.692330,NaN,NaN,Q,5.0,5.0,Q,KCs,KCs-5
TTTGTTGTCTACCCAC-1-GSM5590458_scQueen2,GSM5590458_scQueen2,Q2_sc_TTTGTTGTCTACCCAC-1,Q2,1849.0,999.0,13.142239,NaN,NaN,Q,32.0,32.0,Q,Glial cells,Glial-cells-32
TTTGTTGTCTCACTCG-1-GSM5590458_scQueen2,GSM5590458_scQueen2,Q2_sc_TTTGTTGTCTCACTCG-1,Q2,5251.0,2194.0,4.532470,NaN,NaN,Q,11.0,11.0,Q,KCs,KCs-11


In [8]:
Amel.obs["saturn_batch"] = Amel.obs["sample"].astype(str)

In [9]:
assert not Amel.obs["saturn_label"].isna().any()
Amel.write_h5ad("matrix/Amel_saturn_input.h5ad")